<div style="max-width:100%;box-sizing:border-box;overflow:visible;border-top:4px solid #0f766e;padding:32px 0 20px;margin:0 0 24px">
  <div style="display:block;color:#0f766e;font-size:13px;line-height:1.8;font-weight:700;letter-spacing:0.8px;text-transform:uppercase;margin:0 0 8px">LAB 8 · DATA WAREHOUSING WITH APACHE DORIS</div>
  <div style="color:#17212b;font-size:30px;line-height:1.3;font-weight:750;margin:0 0 10px">Share One Definition, Store Its Result, and Check the Rewrite</div>
</div>

Two teams count "daily orders" from the orders that Level 1 left, and get different numbers. You agree on one definition and publish it as a regular view, store its daily result in an asynchronous materialized view, check the view's refresh and its rows, and read in `EXPLAIN` which queries it answers. Then, on small tables of their own, you watch a stored result go stale and catch up, a synchronous materialized view that every write maintains, and a partitioned materialized view that refreshes one day or both. Plan about 65 minutes.

**Module 8: Views and Materialized Views** · [Lesson](course.md) · [Quiz](quiz08_views_and_materialized_views.ipynb) · [Course README](../../README.md)

## Before you begin

Complete Lab 7 first, in the same database. This Lab reads `orders_current` and `order_events`, the current state and the history of the orders that Lab 7 left, and changes neither. If they are missing, the setup cell stops and says which Lab to finish.

The Lab rebuilds only its own objects, all ending in `_l2`: the view `orders_view_l2`, the materialized view `orders_daily_mv_l2`, and in the practice `orders_region_mv_l2`. Sections 5–8 work on small tables of their own: `orders_refresh_source_l2` with `orders_refresh_view_l2` and `orders_refresh_mv_l2`; `mv_sync_source_l2` with `mv_sync_sum_l2`; and `orders_region_partition_l2` and `customer_region_mv_l2` with `orders_region_partition_mv_l2`. They stay in your lab database after the Lab. Everything runs in the course sandbox; no other container is needed.

In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "dw_course").is_dir())
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from dw_course.docker_runtime import connect_sandbox
from dw_course.runtime import expect, normalized
from dw_course.ui import show_records, show_sql
from dw_course.warehouse import wait_for_mv

lab = connect_sandbox(module=8)

## 1. Two answers to "daily orders"

Operations and Finance both ask for the orders of each day. `orders_current` keeps one row per order, and its `event_time` is the time of the order's latest event: when it was delivered, cancelled, or refunded, not when it was placed. `order_events` keeps every event; an order's version 1 is the event that created it.

The first query groups the current orders by the date of their latest event. The second groups them by the date of the event that created them, and leaves out cancelled orders. Predict how many orders each one counts on January 1, then run the cell.

In [ ]:
lab.sql("""
SELECT DATE(event_time) AS order_date, COUNT(*) AS order_count, SUM(order_amount) AS gross_amount
FROM orders_current
GROUP BY DATE(event_time)
ORDER BY order_date
""", title="Grouped by the date of each order's latest event")
lab.sql("""
SELECT DATE(e.event_time) AS order_date, COUNT(*) AS order_count, SUM(o.order_amount) AS gross_amount
FROM orders_current o
JOIN order_events e ON e.order_id = o.order_id AND e.event_version = 1
WHERE o.status <> 'CANCELLED'
GROUP BY DATE(e.event_time)
ORDER BY order_date
""", title="Grouped by the date each order was created, without cancelled orders");

**Expected result**

Two tables. By the latest event: 2026-01-01 with 7 orders and 950.00, 2026-01-02 with 4 orders and 560.00. By creation, without cancelled orders: 2026-01-01 with 9 orders and 1200.00, 2026-01-02 with 1 order and 110.00.

Both queries run, and both are reasonable SQL. The first counts orders 900001, 900002, and 900003 on January 2, the day they were delivered, cancelled, and refunded, and it counts the cancelled order 900002 and its 200.00. The second counts each order once, on the day it was placed, unless it was cancelled. This course takes the second definition: an order's date is the date it was created, and a cancelled order is not counted. The next step writes it down once, for everyone.

## 2. Publish the definition as a regular view

A regular view saves a query under a name. `orders_view_l2` says what an order is for reporting: its `order_date` is the date of its version 1 event, cancelled orders are left out, and it shows the columns that reports use. Every query of the view runs the saved query on the current tables: the view stores no rows. It is not an access grant either; Module 12 grants access.

The cell first drops this Lab's views, so that it can be rerun.

In [ ]:
lab.execute("DROP MATERIALIZED VIEW IF EXISTS orders_region_mv_l2")
lab.execute("DROP MATERIALIZED VIEW IF EXISTS orders_daily_mv_l2")
lab.execute("DROP VIEW IF EXISTS orders_view_l2")
view_ddl = """CREATE VIEW orders_view_l2 AS
SELECT o.order_id,
       DATE(e.event_time) AS order_date,
       o.customer_id, o.status, o.order_amount, o.paid_amount, o.refund_amount, o.region
FROM orders_current o
JOIN order_events e ON e.order_id = o.order_id AND e.event_version = 1
WHERE o.status <> 'CANCELLED'"""
show_sql("The view's definition", view_ddl)
lab.execute(view_ddl)
lab.sql("SELECT * FROM orders_view_l2 ORDER BY order_id", title="The orders that count, with their order date")
expect(lab.query("SELECT COUNT(*), SUM(order_amount) FROM orders_view_l2"), [(10, "1310.00")],
       title="Ten orders, 1310.00: every order except the cancelled 900002")

**Expected result**

The definition, then ten rows: orders 900001 and 900003–900011, without 900002. Nine have `order_date` 2026-01-01 and 900011 has 2026-01-02; 900001 is DELIVERED, 900003 REFUNDED with 150.00 paid and 150.00 refunded, and the others CREATED. A green check: ten orders, 1310.00.

`order_amount` is the order's pre-tax value, not money received: 900003 keeps its 150.00 although it was refunded. Payments and refunds are measures of their own, `paid_amount` and `refund_amount`.

## 3. Store the daily result in an asynchronous materialized view

Every dashboard that opens the daily totals gets the same answer until the orders change. An asynchronous materialized view stores that answer: it has a query, like a view, and a result, like a table, which a refresh recomputes.

- `BUILD IMMEDIATE` starts the first build when the view is created.
- `REFRESH AUTO` lets Doris decide how much to recompute: the changed partitions when it can tell which ones, everything otherwise.
- `ON MANUAL`: only a `REFRESH MATERIALIZED VIEW` statement starts a refresh. A query does not.

The materialized view uses the same tables, join, and filter as `orders_view_l2`. Its build runs as a task after `CREATE` returns, so the cell waits until `mv_infos` reports the build, then compares the stored rows with the definition.

In [ ]:
mv_ddl = """CREATE MATERIALIZED VIEW orders_daily_mv_l2
BUILD IMMEDIATE
REFRESH AUTO ON MANUAL
AS
SELECT DATE(e.event_time) AS order_date,
       COUNT(*) AS order_count,
       SUM(o.order_amount) AS gross_amount
FROM orders_current o
JOIN order_events e ON e.order_id = o.order_id AND e.event_version = 1
WHERE o.status <> 'CANCELLED'
GROUP BY DATE(e.event_time)"""
show_sql("The materialized view's definition", mv_ddl)
lab.execute(mv_ddl)
mv_state = ("SELECT Name, State, RefreshState, SyncWithBaseTables FROM mv_infos('database'=%s) "
            "WHERE Name = 'orders_daily_mv_l2'")
wait_for_mv(lab, "orders_daily_mv_l2", lambda: lab.query(mv_state, (lab.database,)),
            lambda rows: bool(rows) and rows[0][2] == "SUCCESS", description="The first build of orders_daily_mv_l2")
lab.sql(mv_state, (lab.database,), title="The materialized view's state")
lab.sql("SELECT * FROM orders_daily_mv_l2 ORDER BY order_date", title="The stored daily result")
expect(lab.query("SELECT CAST(order_date AS STRING), order_count, gross_amount FROM orders_daily_mv_l2 ORDER BY order_date"),
       [("2026-01-01", 9, "1200.00"), ("2026-01-02", 1, "110.00")], title="The stored result matches the definition")

**Expected result**

The definition, a progress card that turns green after a few seconds, then the state: `State` NORMAL, `RefreshState` SUCCESS, `SyncWithBaseTables` 1. The stored result has two rows, 2026-01-01 with 9 orders and 1200.00 and 2026-01-02 with 1 order and 110.00, and the check is green.

The three state columns answer different questions: `State` whether the materialized view can be used at all, `RefreshState` whether its last refresh succeeded, and `SyncWithBaseTables` whether its result still matches the base tables. A successful `CREATE` tells none of these; it only accepted the definition.

## 4. Read in EXPLAIN whether a query used the materialized view

Dashboards query `orders_view_l2`, not the materialized view. When a query asks for something that the materialized view stores, Doris can rewrite it to read the stored rows instead of the base tables: transparent rewriting. `EXPLAIN` shows the plan without running the query: which table or materialized view each scan reads, and the materialized views that Doris tried, listed as chosen, usable but not chosen, or not usable with the reason.

The cell explains three queries of the view: the daily totals, the totals per day and region, and the totals of January 1 alone. Predict which ones the materialized view can answer.

In [ ]:
daily_sql = """SELECT order_date, COUNT(*) AS order_count, SUM(order_amount) AS gross_amount
FROM orders_view_l2 GROUP BY order_date"""
scans, views = lab.explain_scan(daily_sql, title="The daily totals of the view")
expect([scan["reads"] for scan in scans], ["orders_daily_mv_l2"], title="The daily query reads orders_daily_mv_l2")
region_sql = """SELECT order_date, region, COUNT(*) AS order_count, SUM(order_amount) AS gross_amount
FROM orders_view_l2 GROUP BY order_date, region"""
scans, views = lab.explain_scan(region_sql, title="The totals per day and region")
expect(sorted(scan["reads"] for scan in scans), ["order_events", "orders_current"],
       title="The query per region reads the base tables")
one_day_sql = """SELECT order_date, COUNT(*) AS order_count, SUM(order_amount) AS gross_amount
FROM orders_view_l2 WHERE order_date = '2026-01-01' GROUP BY order_date"""
scans, views = lab.explain_scan(one_day_sql, title="The totals of January 1")
expect(sorted(scan["reads"] for scan in scans), ["order_events", "orders_current"],
       title="The query of one day reads the base tables too")
lab.sql(daily_sql + " ORDER BY order_date", title="The daily totals, as a dashboard sees them");

**Expected result**

Three plans, each with a green check.

- The daily totals: one scan, which reads `orders_daily_mv_l2`, listed as chosen.
- The totals per day and region: two scans, of `orders_current` and `order_events`, and `orders_daily_mv_l2` not usable: `View struct info is invalid, View dimensions doesn't not cover the query dimensions`.
- The totals of January 1: two scans again, and `orders_daily_mv_l2` not usable: `View struct info is invalid, Rewrite compensate predicate by view fail`. In the whole plan, the scan of `order_events` has the predicate `event_time >= '2026-01-01 00:00:00' AND event_time <= '2026-01-01 23:59:59'`.

Then the daily totals: 9 orders and 1200.00, 1 order and 110.00.

The daily query names only the view, and Doris read the stored result. The query per region cannot use it: the stored rows have no region, and no rewrite can bring back a column that the materialized view did not keep. The query of one day surprises: the stored rows have `order_date`, but Doris turned `DATE(event_time) = '2026-01-01'` into a range on `event_time`, a column that the materialized view does not keep, so it could not apply the filter to the stored rows (Doris 4.1.3). Only `EXPLAIN` shows which queries are rewritten; and no plan says whether an answer is right: step 3 compared the stored rows with the definition.

## 5. Watch a stored result go stale, then catch up

Later Labs read the Level 1 tables, so this step writes to a small table of its own, `orders_refresh_source_l2`, with one order of 100.00. It gets a regular view and a materialized view with the same daily query. Then a second order of 50.00 arrives. Predict both results before you run the cell: the regular view runs its query now; the materialized view, `ON MANUAL`, still holds the result of its last refresh.

In [ ]:
lab.execute("DROP MATERIALIZED VIEW IF EXISTS orders_refresh_mv_l2")
lab.execute("DROP VIEW IF EXISTS orders_refresh_view_l2")
lab.execute("DROP TABLE IF EXISTS orders_refresh_source_l2")
lab.execute("""
CREATE TABLE orders_refresh_source_l2 (
    event_time DATETIME NOT NULL,
    order_id BIGINT NOT NULL,
    order_amount DECIMAL(18,2) NOT NULL
)
DUPLICATE KEY(event_time, order_id)
DISTRIBUTED BY HASH(order_id) BUCKETS 1
PROPERTIES ("replication_num"="1")
""")
lab.insert("orders_refresh_source_l2", ["event_time", "order_id", "order_amount"], [("2026-01-01 12:00:00", 999001, 100)])
lab.execute("""
CREATE VIEW orders_refresh_view_l2 AS
SELECT DATE(event_time) AS order_date, COUNT(*) AS order_count, SUM(order_amount) AS gross_amount
FROM orders_refresh_source_l2 GROUP BY DATE(event_time)
""")
lab.execute("""
CREATE MATERIALIZED VIEW orders_refresh_mv_l2
BUILD IMMEDIATE REFRESH AUTO ON MANUAL
AS SELECT DATE(event_time) AS order_date, COUNT(*) AS order_count, SUM(order_amount) AS gross_amount
FROM orders_refresh_source_l2 GROUP BY DATE(event_time)
""")
wait_for_mv(lab, "orders_refresh_mv_l2",
            lambda: normalized(lab.query("SELECT order_count, gross_amount FROM orders_refresh_mv_l2")),
            lambda rows: rows == [[1, "100.00"]], description="The first build of orders_refresh_mv_l2")
lab.insert("orders_refresh_source_l2", ["event_time", "order_id", "order_amount"], [("2026-01-01 13:00:00", 999002, 50)])
contrast_sql = """
SELECT 'regular view' AS result_source, order_count, gross_amount FROM orders_refresh_view_l2
UNION ALL
SELECT 'stored MV', order_count, gross_amount FROM orders_refresh_mv_l2
ORDER BY result_source
"""
lab.sql(contrast_sql, title="After the write, the regular view and the stored result disagree")
expect(lab.query("SELECT order_count, gross_amount FROM orders_refresh_view_l2"), [(2, "150.00")],
       title="The regular view includes the new order")
expect(lab.query("SELECT order_count, gross_amount FROM orders_refresh_mv_l2"), [(1, "100.00")],
       title="The materialized view still holds its last result")

**Expected result**

A progress card for the first build, then a table with two rows, `regular view` with 2 orders and 150.00 and `stored MV` with 1 order and 100.00, and two green checks. The materialized view still holds what its first build computed.

Now refresh it. `REFRESH ... COMPLETE` recomputes the whole result; the statement returns when the task is submitted, so the cell waits for the new result before it compares. The regular view needs no refresh: it has no stored result.

In [ ]:
lab.execute("REFRESH MATERIALIZED VIEW orders_refresh_mv_l2 COMPLETE")
wait_for_mv(lab, "orders_refresh_mv_l2",
            lambda: normalized(lab.query("SELECT order_count, gross_amount FROM orders_refresh_mv_l2")),
            lambda rows: rows == [[2, "150.00"]], description="The manual refresh of orders_refresh_mv_l2")
lab.sql(contrast_sql, title="After the refresh, both results agree")
expect(lab.query("SELECT order_count, gross_amount FROM orders_refresh_mv_l2"), [(2, "150.00")],
       title="The stored result caught up after the refresh")
expect(lab.query("SELECT COUNT(*), SUM(order_amount) FROM orders_current"), [(11, "1510.00")],
       title="Lab 7's current orders are unchanged")

**Expected result**

A progress card, then both rows with 2 orders and 150.00, and two green checks: the stored result caught up, and Lab 7's `orders_current` still holds 11 orders and 1510.00.

A refresh statement that succeeded, a result as fresh as the dashboard needs, and a query rewritten to read the result are three different checks: step 4 checked the rewrite, this step the refresh.

## 6. A synchronous materialized view, maintained by every write

A synchronous materialized view belongs to one table, and each load into the table also updates it, in the same transaction: there is no refresh to run. Build one that sums the amount per day on `mv_sync_source_l2`, a Duplicate Key table of its own. Its first build is a background job, so the cell records the time before `CREATE` and waits until `SHOW ALTER TABLE MATERIALIZED VIEW` reports this build `FINISHED`. Then it loads a second order and explains a daily sum of the base table.

Predict: does the second order need a refresh before the daily sum includes it?

In [ ]:
from dw_course.streaming import wait_for
import pymysql

lab.execute("DROP TABLE IF EXISTS mv_sync_source_l2")
lab.execute("""
CREATE TABLE mv_sync_source_l2 (
    order_date DATE NOT NULL, order_id BIGINT NOT NULL,
    amount DECIMAL(18,2) NOT NULL
) DUPLICATE KEY(order_date, order_id)
DISTRIBUTED BY HASH(order_id) BUCKETS 1
PROPERTIES ("replication_num"="1")
""")
lab.insert("mv_sync_source_l2", ["order_date", "order_id", "amount"], [("2026-02-01", 8101, 100)])
created_after = str(lab.query("SELECT NOW()")[0][0])
lab.execute("""
CREATE MATERIALIZED VIEW mv_sync_sum_l2 AS
SELECT order_date AS mv_day, SUM(amount) AS mv_amount
FROM mv_sync_source_l2 GROUP BY order_date
""")
build_job = (f"SHOW ALTER TABLE MATERIALIZED VIEW FROM {lab.database} WHERE TableName = 'mv_sync_source_l2' "
             f"AND RollupIndexName = 'mv_sync_sum_l2' AND CreateTime >= '{created_after}'")

def build_state():
    with lab.connection.cursor(pymysql.cursors.DictCursor) as cursor:
        cursor.execute(build_job)
        jobs = cursor.fetchall()
    if jobs and jobs[-1]["State"] == "CANCELLED":
        raise RuntimeError(f"The build of mv_sync_sum_l2 was cancelled: {jobs[-1]['Msg']}")
    return jobs[-1]["State"] if jobs else None

wait_for(build_state, lambda state: state == "FINISHED", description="The first build of mv_sync_sum_l2", timeout=120)
lab.sql(build_job, title="This build of mv_sync_sum_l2",
        columns=["JobId", "TableName", "RollupIndexName", "CreateTime", "FinishTime", "State"])
lab.insert("mv_sync_source_l2", ["order_date", "order_id", "amount"], [("2026-02-01", 8102, 50)])
scans, views = lab.explain_scan("SELECT order_date, SUM(amount) FROM mv_sync_source_l2 GROUP BY order_date",
                                title="A daily sum of the base table")
expect([scan["reads"] for scan in scans], ["mv_sync_sum_l2"], title="The plan reads mv_sync_sum_l2, not the table's rows")
expect(lab.query("SELECT SUM(amount) FROM mv_sync_source_l2"), [("150.00",)], title="The second order counts without a refresh")

**Expected result**

A progress card, the build job of this run `FINISHED`, then the plan: one scan of `mv_sync_source_l2` that reads `mv_sync_sum_l2`, the materialized view listed as chosen, and two green checks, the plan reads `mv_sync_sum_l2` and the daily sum is 150.00.

The second order was in `mv_sync_sum_l2` as soon as its load committed, because the load updated it. The cost moved to the writes, which now maintain two sets of rows; every synchronous materialized view on a table makes its loads do more work.

## 7. Refresh only the changed partition

A partitioned asynchronous materialized view can refresh one partition when only one partition of its source changed. `orders_region_partition_l2` has two daily partitions, and it is joined with a small customer table, `customer_region_mv_l2`, that has no partitions; the materialized view `PARTITION BY (order_date)` groups the orders by date and region, so each of its partitions follows one fact date.

Predict: a new order on February 2 should refresh February 2's partition and leave February 1's. The cell compares each partition's `PartitionId` before and after: a partition that was published again gets a new one, even when its `VisibleVersion` stays 2. Then it reads the refresh task's record.

In [ ]:
lab.execute("DROP MATERIALIZED VIEW IF EXISTS orders_region_partition_mv_l2")
lab.execute("DROP TABLE IF EXISTS orders_region_partition_l2")
lab.execute("DROP TABLE IF EXISTS customer_region_mv_l2")
lab.execute("""
CREATE TABLE orders_region_partition_l2 (
    order_date DATE NOT NULL, order_id BIGINT NOT NULL,
    customer_id BIGINT NOT NULL, amount DECIMAL(18,2) NOT NULL
) DUPLICATE KEY(order_date, order_id)
PARTITION BY RANGE(order_date) (
    PARTITION p20260201 VALUES [("2026-02-01"), ("2026-02-02")),
    PARTITION p20260202 VALUES [("2026-02-02"), ("2026-02-03"))
)
DISTRIBUTED BY HASH(order_id) BUCKETS 1 PROPERTIES ("replication_num"="1")
""")
lab.execute("""
CREATE TABLE customer_region_mv_l2 (customer_id BIGINT NOT NULL, region VARCHAR(16) NOT NULL)
UNIQUE KEY(customer_id) DISTRIBUTED BY HASH(customer_id) BUCKETS 1
PROPERTIES ("replication_num"="1")
""")
lab.insert("customer_region_mv_l2", ["customer_id", "region"], [(1,"EAST"),(2,"WEST")])
lab.insert("orders_region_partition_l2", ["order_date","order_id","customer_id","amount"],
           [("2026-02-01",8201,1,100),("2026-02-02",8202,2,200)])
lab.execute("""
CREATE MATERIALIZED VIEW orders_region_partition_mv_l2
BUILD IMMEDIATE REFRESH AUTO ON MANUAL
PARTITION BY (order_date)
DISTRIBUTED BY HASH(order_date) BUCKETS 1
PROPERTIES ("replication_num"="1", "refresh_partition_num"="10")
AS SELECT o.order_date, c.region, COUNT(*) AS order_count, SUM(o.amount) AS gross_amount
FROM orders_region_partition_l2 o JOIN customer_region_mv_l2 c ON o.customer_id=c.customer_id
GROUP BY o.order_date, c.region
""")
region_result = "SELECT CAST(order_date AS STRING), region, order_count, gross_amount FROM orders_region_partition_mv_l2 ORDER BY order_date, region"
wait_for_mv(lab, "orders_region_partition_mv_l2", lambda: normalized(lab.query(region_result)),
            lambda rows: rows == [["2026-02-01","EAST",1,"100.00"],["2026-02-02","WEST",1,"200.00"]],
            description="The first build of orders_region_partition_mv_l2", timeout=120)

def mv_partition_publications():
    frame = lab.sql("SHOW PARTITIONS FROM orders_region_partition_mv_l2", title="The materialized view's partitions",
                    columns=["PartitionId", "PartitionName", "VisibleVersion", "VisibleVersionTime"])
    return {row.PartitionName: (row.PartitionId, row.VisibleVersion) for row in frame.itertuples()}

versions_before = mv_partition_publications()
lab.insert("orders_region_partition_l2", ["order_date","order_id","customer_id","amount"], [("2026-02-02",8203,2,50)])
lab.execute("REFRESH MATERIALIZED VIEW orders_region_partition_mv_l2 AUTO")
wait_for_mv(lab, "orders_region_partition_mv_l2", lambda: normalized(lab.query(region_result)),
            lambda rows: rows == [["2026-02-01","EAST",1,"100.00"],["2026-02-02","WEST",2,"250.00"]],
            description="The refresh after the new order on February 2", timeout=120)
versions_after_fact = mv_partition_publications()
expect(sum(versions_after_fact[name] != version for name, version in versions_before.items()), 1,
       title="Exactly one partition was published again")
refresh_tasks = ("SELECT TaskId, Status, RefreshMode, NeedRefreshPartitions FROM tasks('type'='mv') "
                 "WHERE MvName = 'orders_region_partition_mv_l2' AND MvDatabaseName = %s ORDER BY CreateTime DESC LIMIT 3")
lab.sql(refresh_tasks, (lab.database,), title="The refresh tasks, newest first");

**Expected result**

A progress card and the two partitions of the materialized view; after the new order, a second progress card and the partitions again, where `p_20260202_20260203` has a new `PartitionId` and `p_20260201_20260202` the same one as before. A green check: exactly one partition was published again. The tasks, newest first: the refresh with `RefreshMode` PARTIAL and `NeedRefreshPartitions` `["p_20260202_20260203"]`, then the first build with both partitions.

## 8. A dimension change can widen the refresh

Now change customer 1's region from EAST to NORTH. The customer table has no partitions, so nothing maps this change to one date. `REFRESH ... AUTO` has to assume that every partition of the materialized view may have changed, and this refresh publishes both partitions, even the one whose numbers stay the same.

A new `PartitionId` shows that a partition was published again, not how much work that took. The task's record shows which partitions were recomputed; a Query Profile would show the work.

In [ ]:
lab.insert("customer_region_mv_l2", ["customer_id","region"], [(1,"NORTH")])
lab.execute("REFRESH MATERIALIZED VIEW orders_region_partition_mv_l2 AUTO")
wait_for_mv(lab, "orders_region_partition_mv_l2", lambda: normalized(lab.query(region_result)),
            lambda rows: rows == [["2026-02-01","NORTH",1,"100.00"],["2026-02-02","WEST",2,"250.00"]],
            description="The refresh after the region change of customer 1", timeout=120)
versions_after_dimension = mv_partition_publications()
expect(sum(versions_after_dimension[name] != version for name, version in versions_after_fact.items()), 2,
       title="The dimension change published both partitions")
show_records("Refresh comparison", [
    {"change":"one fact date", "partitions_changed":1, "other_day_amount":"100.00"},
    {"change":"unpartitioned dimension", "partitions_changed":2, "other_day_amount":"250.00"},
])
lab.sql(refresh_tasks, (lab.database,), title="The refresh tasks, newest first");

**Expected result**

A progress card, then the partitions: both have a new `PartitionId`, and the check is green. The comparison: one fact date published one partition; the dimension change published both, although February 2's 250.00 did not change. The newest task has `RefreshMode` COMPLETE and both partitions in `NeedRefreshPartitions`.

The refresh scope comes from what Doris can map to the materialized view's partitions. A fact partition maps to one date; a change in an unpartitioned table maps to none, so all of them are recomputed.

## 9. Practice: daily orders per region

A regional dashboard asks for the orders of each day and region. Step 4 showed that `orders_daily_mv_l2` cannot answer: its rows have no region. Create a second materialized view, `orders_region_mv_l2`, at the grain that the dashboard needs, with the same tables, join, and filter as `orders_view_l2`. Wait for its first build, compare its rows with the view's totals per day and region, then explain step 4's query per region again. Leave `orders_daily_mv_l2` as it is: its consumers still read it.

In [ ]:
# Your code here

The next cell is a reference solution; it is collapsed. Expand it to compare, or run it.

In [ ]:
lab.execute("DROP MATERIALIZED VIEW IF EXISTS orders_region_mv_l2")
lab.execute("""CREATE MATERIALIZED VIEW orders_region_mv_l2
BUILD IMMEDIATE REFRESH AUTO ON MANUAL
AS
SELECT DATE(e.event_time) AS order_date, o.region,
       COUNT(*) AS order_count, SUM(o.order_amount) AS gross_amount
FROM orders_current o
JOIN order_events e ON e.order_id = o.order_id AND e.event_version = 1
WHERE o.status <> 'CANCELLED'
GROUP BY DATE(e.event_time), o.region""")
stored = "SELECT CAST(order_date AS STRING) AS order_date, region, order_count, gross_amount FROM orders_region_mv_l2 ORDER BY order_date, region"
from_view = """SELECT CAST(order_date AS STRING), region, COUNT(*), SUM(order_amount)
FROM orders_view_l2 GROUP BY order_date, region ORDER BY order_date, region"""
wait_for_mv(lab, "orders_region_mv_l2", lambda: normalized(lab.query(stored)), lambda rows: len(rows) == 3,
            description="The first build of orders_region_mv_l2")
lab.sql(stored, title="The stored totals per day and region")
expect(lab.query(stored), lab.query(from_view), title="The stored rows equal the view's totals")
scans, views = lab.explain_scan(region_sql, title="The totals per day and region, again")
expect([scan["reads"] for scan in scans], ["orders_region_mv_l2"], title="The query per region now reads orders_region_mv_l2")

**Expected result**

A progress card, then three stored rows, (2026-01-01, EAST, 5, 720.00), (2026-01-01, WEST, 4, 480.00), and (2026-01-02, EAST, 1, 110.00), and a green check: they equal the view's totals. The plan reads `orders_region_mv_l2`, listed as chosen, and `orders_daily_mv_l2` is still not usable for this query; the last check is green.

The new materialized view keeps a finer grain than `orders_daily_mv_l2`, so it stores more rows, and it is one more refresh to schedule and check.

## You are done when

- you can say why two reasonable queries counted 7 and 9 orders on January 1, and which definition `orders_view_l2` publishes;
- `orders_daily_mv_l2` reports `RefreshState` SUCCESS, its rows equal the definition, and you can read in `EXPLAIN` which queries it answers and why the others cannot use it;
- you have seen a manual-refresh result go stale and catch up, a synchronous materialized view that a load updates, and a refresh of one partition against a refresh of both.

## Next

Open [Quiz 8: Views and Materialized Views](quiz08_views_and_materialized_views.ipynb), then continue to [Module 9: Warehouse Modeling and Joins](../module09-modeling-and-joins/course.md).